# Chapter 6 — Command Pattern
## *Encapsulating Invocation*

**Intent:** Encapsulate a request as an object, letting you parameterize clients with different requests, queue or log requests, and support undoable operations.

### Key roles
| Role | Responsibility |
|---|---|
| **Command** | Declares `execute()` (and optionally `undo()`) |
| **ConcreteCommand** | Binds a receiver with an action |
| **Invoker** | Holds and fires commands (e.g., remote control button) |
| **Receiver** | Knows how to perform the actual work |
| **Client** | Creates ConcreteCommands and sets their receivers |

### Book context
A home-automation remote control with 7 programmable on/off slots and an undo button.

In [ ]:
from abc import ABC, abstractmethod

# ── Command interface ────────────────────────────────────────────────────────
class Command(ABC):
    @abstractmethod
    def execute(self): ...
    @abstractmethod
    def undo(self): ...

In [ ]:
# ── Receivers ────────────────────────────────────────────────────────────────
class Light:
    def __init__(self, location: str):
        self._location = location
        self._on = False

    def on(self):
        self._on = True
        print(f"{self._location} light is ON")

    def off(self):
        self._on = False
        print(f"{self._location} light is OFF")


class CeilingFan:
    HIGH, MEDIUM, LOW, OFF = 3, 2, 1, 0

    def __init__(self, location: str):
        self._location = location
        self._speed = self.OFF

    def high(self):
        self._speed = self.HIGH
        print(f"{self._location} fan → HIGH")

    def medium(self):
        self._speed = self.MEDIUM
        print(f"{self._location} fan → MEDIUM")

    def off(self):
        self._speed = self.OFF
        print(f"{self._location} fan → OFF")

    @property
    def speed(self): return self._speed

In [ ]:
# ── Concrete Commands ─────────────────────────────────────────────────────────
class LightOnCommand(Command):
    def __init__(self, light: Light):
        self._light = light
    def execute(self): self._light.on()
    def undo(self):    self._light.off()

class LightOffCommand(Command):
    def __init__(self, light: Light):
        self._light = light
    def execute(self): self._light.off()
    def undo(self):    self._light.on()


class CeilingFanHighCommand(Command):
    def __init__(self, fan: CeilingFan):
        self._fan = fan
        self._prev_speed = 0
    def execute(self):
        self._prev_speed = self._fan.speed
        self._fan.high()
    def undo(self):
        {0: self._fan.off, 1: self._fan.medium, 3: self._fan.high}\
            .get(self._prev_speed, self._fan.off)()


class NoCommand(Command):   # Null Object pattern — avoids None checks
    def execute(self): pass
    def undo(self):    pass


# Macro command — composite of commands
class MacroCommand(Command):
    def __init__(self, commands: list[Command]):
        self._commands = commands
    def execute(self):
        for c in self._commands: c.execute()
    def undo(self):
        for c in reversed(self._commands): c.undo()

In [ ]:
# ── Invoker: Remote Control ───────────────────────────────────────────────────
class RemoteControl:
    SLOTS = 7

    def __init__(self):
        no_cmd = NoCommand()
        self._on_commands  = [no_cmd] * self.SLOTS
        self._off_commands = [no_cmd] * self.SLOTS
        self._last: Command = no_cmd

    def set_command(self, slot: int, on_cmd: Command, off_cmd: Command):
        self._on_commands[slot]  = on_cmd
        self._off_commands[slot] = off_cmd

    def on_button(self, slot: int):
        self._on_commands[slot].execute()
        self._last = self._on_commands[slot]

    def off_button(self, slot: int):
        self._off_commands[slot].execute()
        self._last = self._off_commands[slot]

    def undo(self):
        print("[UNDO]")
        self._last.undo()

In [ ]:
# ── Demo ─────────────────────────────────────────────────────────────────────
remote = RemoteControl()

living_light = Light("Living Room")
kitchen_light = Light("Kitchen")
fan = CeilingFan("Living Room")

remote.set_command(0, LightOnCommand(living_light),  LightOffCommand(living_light))
remote.set_command(1, LightOnCommand(kitchen_light), LightOffCommand(kitchen_light))
remote.set_command(2, CeilingFanHighCommand(fan),    NoCommand())

remote.on_button(0)
remote.on_button(1)
remote.on_button(2)
remote.off_button(1)
print()
remote.undo()   # re-enables kitchen light
print()

# Macro: lights-out party mode
party_off = MacroCommand([
    LightOffCommand(living_light),
    LightOffCommand(kitchen_light),
])
remote.set_command(6, NoCommand(), party_off)
print("--- Party mode OFF ---")
remote.off_button(6)
print("--- Undo party mode ---")
remote.undo()

## Pythonic variant — using callables
In Python, any callable is a command; `functools.partial` creates parameterized commands without boilerplate classes.

In [ ]:
from functools import partial
from collections import deque

class FunctionalRemote:
    def __init__(self):
        self._history: deque = deque(maxlen=20)

    def execute(self, do, undo):
        do()
        self._history.append(undo)

    def undo(self):
        if self._history:
            self._history.pop()()

light = Light("Bedroom")
remote2 = FunctionalRemote()
remote2.execute(light.on, light.off)
remote2.undo()

## Interview cheat-sheet

| Question | Answer |
|---|---|
| What problems does Command solve? | Decouples sender from receiver; enables undo/redo, queuing, logging, transactions. |
| Null Object pattern? | `NoCommand` avoids None checks throughout the invoker. |
| Real-world uses? | GUI action history, job queues (Celery tasks), database transactions, HTTP request objects. |
| Macro Command? | A Command that holds a list of commands — composite + command combined. |

**Pattern family:** Behavioral